# Trabajo Práctico Nro. 1 
### Ing. Javier Ouret - 2025 - UCA - Facultad de Ingeniería
## Ejemplos de Servidores TCP y Clientes TCP usando bash kernel para su compilación.
## A) Servidor Simple y Servidor Concurrente

**NOTA: Cerrar las ventanas de las Terminales al finalizar su uso**

## Servidor TCP

Envío el código a un archivo:

In [1]:
echo '#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <unistd.h>
#include <netinet/in.h>
#include <arpa/inet.h>

#define MAX 1024
#define PORT 8080

int main() {
    int server_fd, client_fd;
    struct sockaddr_in server_addr, client_addr;
    char buffer[MAX];
    char client_name[50];

    // Crear socket
    server_fd = socket(AF_INET, SOCK_STREAM, 0);
    if (server_fd == -1) {
        perror("Socket no pudo abrirse ... error ...");
        exit(1);
    }

    // Configurar estructura
    server_addr.sin_family = AF_INET;
    server_addr.sin_addr.s_addr = INADDR_ANY;
    server_addr.sin_port = htons(PORT);
    memset(&(server_addr.sin_zero), 0, 8);

    // Asociar dirección y puerto
    if (bind(server_fd, (struct sockaddr*)&server_addr, sizeof(server_addr)) != 0) {
        perror("Bind falló");
        exit(1);
    }

    // Escuchar conexiones
    if (listen(server_fd, 1) != 0) {
        perror("Listen con error...");
        exit(1);
    }

    printf("Servidor esperando conexión en el puerto %d...\n", PORT);
    socklen_t addr_size = sizeof(client_addr);
    client_fd = accept(server_fd, (struct sockaddr*)&client_addr, &addr_size);
    if (client_fd < 0) {
        perror("Falló al aceptar conexión");
        exit(1);
    }

    // Recibir nombre del cliente
    recv(client_fd, client_name, sizeof(client_name), 0);
    printf("Cliente conectado: %s\n", client_name);

    while (1) {
        // Recibir mensaje del cliente
        memset(buffer, 0, MAX);
        int bytes = recv(client_fd, buffer, MAX, 0);
        if (bytes <= 0) {
            printf("Cliente desconectado.\n");
            break;
        }

        if (strncmp(buffer, "FIN", 3) == 0) {
            printf("%s ha finalizado la conexión.\n", client_name);
            break;
        }

        printf("%s: %s", client_name, buffer);

        // Enviar respuesta
        printf("Servidor: ");
        memset(buffer, 0, MAX);
        fgets(buffer, MAX, stdin);
        send(client_fd, buffer, strlen(buffer), 0);

        if (strncmp(buffer, "FIN", 3) == 0) {
            printf("Servidor finaliza la conexión.\n");
            break;
        }
    }

    close(client_fd);
    close(server_fd);
    return 0;
}
' > servidor_tcp_bash.c

Compilo el código usando Bash C:

In [2]:
gcc --version

gcc (Ubuntu 11.4.0-1ubuntu1~22.04) 11.4.0
Copyright (C) 2021 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.



In [3]:
gcc servidor_tcp_bash.c -o servidor_tcp_bash

Verifico que el ejecutable esté disponible:

In [4]:
ls

 cliente                                servidor
 cliente.c                              servidor.c
'Cliente Servidor TCP con bash.ipynb'   servidor_tcp_bash
 cliente_tcp_bash                       servidor_tcp_bash.c
 cliente_tcp_bash.c                     servidor_tcp_c.ipynb
 cliente_tcp_c.ipynb                    servidor_tcp_concurrente_bash
'prueba de C con bash.ipynb'            servidor_tcp_concurrente_bash.c
'prueba de C.ipynb'


La terminal de Jupyter puede funcionar mal. Para ejecutarlo correctamente abro ventana por afuera del Jupyter Notebook:

In [5]:
gnome-terminal -- bash -c "./servidor_tcp_bash; exec bash"

## Cliente TCP

Envío el código a un archivo:

In [6]:
echo '#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <unistd.h>
#include <netinet/in.h>
#include <arpa/inet.h>

#define MAX 1024
#define PORT 8080

int main() {
    int sockfd;
    struct sockaddr_in server_addr;
    char buffer[MAX];
    char name[50];

    // Crear socket
    sockfd = socket(AF_INET, SOCK_STREAM, 0);
    if (sockfd == -1) {
        perror("Socket falló");
        exit(1);
    }

    server_addr.sin_family = AF_INET;
    server_addr.sin_port = htons(PORT);
    inet_pton(AF_INET, "127.0.0.1", &server_addr.sin_addr);
    memset(&(server_addr.sin_zero), 0, 8);

    // Conectar al servidor
    if (connect(sockfd, (struct sockaddr*)&server_addr, sizeof(server_addr)) != 0) {
        perror("Conexión falló");
        exit(1);
    }

    // Enviar nombre
    printf("Ingresar nombre: ");
    fgets(name, sizeof(name), stdin);
    name[strcspn(name, "\n")] = 0; // Eliminar salto de línea
    send(sockfd, name, strlen(name), 0);

    while (1) {
        // Enviar mensaje
        printf("%s: ", name);
        memset(buffer, 0, MAX);
        fgets(buffer, MAX, stdin);
        send(sockfd, buffer, strlen(buffer), 0);

        if (strncmp(buffer, "FIN", 3) == 0) {
            printf("Terminando la conexión...\n");
            break;
        }

        // Recibir respuesta
        memset(buffer, 0, MAX);
        int bytes = recv(sockfd, buffer, MAX, 0);
        if (bytes <= 0 || strncmp(buffer, "FIN", 3) == 0) {
            printf("Servidor finalizó la conexión.\n");
            break;
        }

        printf("Servidor: %s", buffer);
    }

    close(sockfd);
    return 0;
}' > cliente_tcp_bash.c

Compilo el código usando Bash C:

In [7]:
gcc --version

gcc (Ubuntu 11.4.0-1ubuntu1~22.04) 11.4.0
Copyright (C) 2021 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.



gcc cliente_tcp_bash.c -o cliente_tcp_bash

Verifico que el ejecutable esté disponible:

In [8]:
ls

 cliente                                servidor
 cliente.c                              servidor.c
'Cliente Servidor TCP con bash.ipynb'   servidor_tcp_bash
 cliente_tcp_bash                       servidor_tcp_bash.c
 cliente_tcp_bash.c                     servidor_tcp_c.ipynb
 cliente_tcp_c.ipynb                    servidor_tcp_concurrente_bash
'prueba de C con bash.ipynb'            servidor_tcp_concurrente_bash.c
'prueba de C.ipynb'


Verifico si funciona:

Para ejecutarlo correctamente abro ventana terminal por afuera del Jupyter Notebook (esta ventana permanecerá abierta):

In [17]:
gnome-terminal -- bash -c "./cliente_tcp_bash; exec bash"

Si abro la ventana terminal de esta forma se cerrará al finalizar la ejecución del cliente:

In [18]:
gnome-terminal -- bash -c "./cliente_tcp_bash; echo 'Presionar ENTER para cerrar...'; read"

## Servidor TCP Concurrente

In [ ]:
echo '#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <unistd.h>
#include <netinet/in.h>
#include <arpa/inet.h>
#include <signal.h>

#define MAX 1024
#define PORT 8080

void handle_client(int client_fd, struct sockaddr_in client_addr) {
    char buffer[MAX];
    char name[50] = "Cliente";
    char client_ip[INET_ADDRSTRLEN];
    char response[MAX + 100];
    pid_t pid = getpid();

    inet_ntop(AF_INET, &(client_addr.sin_addr), client_ip, INET_ADDRSTRLEN);
    int client_port = ntohs(client_addr.sin_port);

    printf("[PID %d] Conexión desde %s:%d\n", pid, client_ip, client_port);

    // Recibir nombre del cliente
    recv(client_fd, name, sizeof(name), 0);
    printf("[PID %d] Cliente: %s\n", pid, name);

    while (1) {
        memset(buffer, 0, MAX);
        int bytes = recv(client_fd, buffer, MAX, 0);
        if (bytes <= 0) {
            printf("[PID %d] %s se desconectó.\n", pid, name);
            break;
        }

        // Verificar si cliente quiere finalizar
        if (strncmp(buffer, "FIN", 3) == 0) {
            printf("[PID %d] %s finalizó la conexión.\n", pid, name);
            break;
        }

        printf("[%s | PID %d]: %s", name, pid, buffer);

        // Preparar respuesta automática
        snprintf(response, sizeof(response),
                 "Este es el mensaje recibido por el servidor: %s", buffer);
        send(client_fd, response, strlen(response), 0);
    }

    close(client_fd);
    printf("[PID %d] Proceso hijo termina.\n", pid);
    exit(0);
}

int main() {
    int server_fd, client_fd;
    struct sockaddr_in server_addr, client_addr;

    signal(SIGCHLD, SIG_IGN); // Evita procesos zombies

    server_fd = socket(AF_INET, SOCK_STREAM, 0);
    if (server_fd == -1) {
        perror("Error al crear socket");
        exit(1);
    }

    server_addr.sin_family = AF_INET;
    server_addr.sin_addr.s_addr = INADDR_ANY;
    server_addr.sin_port = htons(PORT);
    memset(&(server_addr.sin_zero), 0, 8);

    if (bind(server_fd, (struct sockaddr*)&server_addr, sizeof(server_addr)) != 0) {
        perror("Error en bind");
        exit(1);
    }

    if (listen(server_fd, 5) != 0) {
        perror("Error en listen");
        exit(1);
    }

    printf("Servidor concurrente activo en puerto %d...\n", PORT);

    socklen_t len = sizeof(client_addr);
    while (1) {
        client_fd = accept(server_fd, (struct sockaddr*)&client_addr, &len);
        if (client_fd < 0) {
            perror("Fallo en accept");
            continue;
        }

        pid_t pid = fork();
        if (pid == 0) {
            close(server_fd);
            handle_client(client_fd, client_addr);
        } else if (pid > 0) {
            printf("[PID %d] Cliente conectado. Proceso hijo PID %d\n", getpid(), pid);
            close(client_fd);
        } else {
            perror("Error en fork");
            close(client_fd);
        }
    }

    close(server_fd);
    return 0;
}' > servidor_tcp_concurrente_bash.c

Compilo el código usando Bash C y verifico que el ejecutable esté disponible

In [ ]:
gcc servidor_tcp_concurrente_bash.c -o servidor_tcp_concurrente_bash | ls


In [ ]:
gnome-terminal -- bash -c "./servidor_tcp_concurrente_bash; echo 'Presionar ENTER para cerrar...'; read"

**Ejercicios a realizar durante la clase. Incluir los códigos dentro de este mismo Notebook**

A-1) Convertir el servidor simple a servidor iterativo.   
A-2) Convertir el cliente y el servidor a un cliente UDP que utilice connect() y un servidor iterativo con UDP.   
A-3) Mejorar la aplicación de chat para que identifique a los clientes conectados.   
A-4) Abrir varios clientes y hacer que los clientes manden mensajes en forma automática al servidor concurrente para observar el comportamiento del fork().   

---
## A-1) Servidor Iterativo TCP

Un **servidor iterativo** atiende a los clientes **de a uno por vez**: acepta una conexión, la procesa completamente, la cierra, y recién entonces acepta la siguiente.

A diferencia del servidor simple (que solo acepta una conexión y termina), el iterativo tiene un bucle externo que sigue aceptando nuevas conexiones indefinidamente.

**Diferencia clave con el servidor concurrente:** no usa `fork()`. Atiende un cliente a la vez — si hay otro esperando, espera en la cola de `listen()`.


In [ ]:
cat > servidor_iterativo.c << 'EOF'
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <unistd.h>
#include <netinet/in.h>
#include <arpa/inet.h>

#define PORT 8081
#define MAX 1024

int main() {
    int server_fd, client_fd;
    struct sockaddr_in server_addr, client_addr;
    socklen_t addr_len = sizeof(client_addr);
    char buffer[MAX];

    server_fd = socket(AF_INET, SOCK_STREAM, 0);

    server_addr.sin_family = AF_INET;
    server_addr.sin_addr.s_addr = INADDR_ANY;
    server_addr.sin_port = htons(PORT);

    bind(server_fd, (struct sockaddr*)&server_addr, sizeof(server_addr));
    listen(server_fd, 5);

    printf("Servidor iterativo escuchando en puerto %d...\n", PORT);

    // Bucle externo: acepta clientes de a uno por vez
    while (1) {
        client_fd = accept(server_fd, (struct sockaddr*)&client_addr, &addr_len);
        printf("Cliente conectado: %s\n", inet_ntoa(client_addr.sin_addr));

        // Bucle interno: atiende al cliente actual
        while (1) {
            memset(buffer, 0, MAX);
            int bytes = recv(client_fd, buffer, MAX, 0);
            if (bytes <= 0) {
                printf("Cliente desconectado.\n");
                break;
            }
            printf("Recibido: %s", buffer);
            send(client_fd, buffer, bytes, 0); // eco: devuelve lo mismo
        }

        close(client_fd);
        // vuelve al bucle externo y espera el siguiente cliente
    }

    close(server_fd);
    return 0;
}
EOF
gcc servidor_iterativo.c -o servidor_iterativo && echo "Compilado OK"

In [ ]:
gnome-terminal -- bash -c "./servidor_iterativo; echo 'Presionar ENTER para cerrar...'; read"

---
## A-2) Cliente y Servidor UDP con connect()

**UDP** es un protocolo sin conexión: no hay `accept()`, no hay estado de conexión, y los mensajes pueden llegar en cualquier orden o no llegar.

**`connect()` en UDP** no establece una conexión real, sino que fija la dirección de destino del socket. Ventaja: se puede usar `send()`/`recv()` en lugar de `sendto()`/`recvfrom()`.

**Servidor iterativo UDP:** espera datagramas con `recvfrom()`, procesa uno a la vez, responde con `sendto()`.


In [ ]:
cat > servidor_udp.c << 'EOF'
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <unistd.h>
#include <netinet/in.h>
#include <arpa/inet.h>

#define PORT 9090
#define MAX 1024

int main() {
    int sockfd;
    struct sockaddr_in server_addr, client_addr;
    socklen_t addr_len = sizeof(client_addr);
    char buffer[MAX];

    // SOCK_DGRAM = UDP (sin conexión)
    sockfd = socket(AF_INET, SOCK_DGRAM, 0);

    server_addr.sin_family = AF_INET;
    server_addr.sin_addr.s_addr = INADDR_ANY;
    server_addr.sin_port = htons(PORT);

    bind(sockfd, (struct sockaddr*)&server_addr, sizeof(server_addr));

    printf("Servidor UDP iterativo escuchando en puerto %d...\n", PORT);

    // No hay accept() en UDP. Simplemente esperamos datagramas.
    while (1) {
        memset(buffer, 0, MAX);
        int bytes = recvfrom(sockfd, buffer, MAX, 0,
                             (struct sockaddr*)&client_addr, &addr_len);
        printf("Mensaje de %s: %s", inet_ntoa(client_addr.sin_addr), buffer);

        // Responde al mismo cliente que envió el mensaje
        sendto(sockfd, buffer, bytes, 0,
               (struct sockaddr*)&client_addr, addr_len);
    }

    close(sockfd);
    return 0;
}
EOF
gcc servidor_udp.c -o servidor_udp && echo "Compilado OK"

In [ ]:
cat > cliente_udp.c << 'EOF'
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <unistd.h>
#include <netinet/in.h>
#include <arpa/inet.h>

#define PORT 9090
#define MAX 1024

int main() {
    int sockfd;
    struct sockaddr_in server_addr;
    char buffer[MAX];

    // SOCK_DGRAM = UDP
    sockfd = socket(AF_INET, SOCK_DGRAM, 0);

    server_addr.sin_family = AF_INET;
    server_addr.sin_port = htons(PORT);
    inet_pton(AF_INET, "127.0.0.1", &server_addr.sin_addr);

    // connect() en UDP: fija el destino. Permite usar send()/recv()
    // en lugar de sendto()/recvfrom(). No establece conexión real.
    connect(sockfd, (struct sockaddr*)&server_addr, sizeof(server_addr));

    printf("Cliente UDP listo. Escribí mensajes (FIN para salir):\n");

    while (1) {
        printf("> ");
        fgets(buffer, MAX, stdin);

        if (strncmp(buffer, "FIN", 3) == 0) break;

        send(sockfd, buffer, strlen(buffer), 0);

        memset(buffer, 0, MAX);
        recv(sockfd, buffer, MAX, 0);
        printf("Eco del servidor: %s", buffer);
    }

    close(sockfd);
    return 0;
}
EOF
gcc cliente_udp.c -o cliente_udp && echo "Compilado OK"

In [ ]:
gnome-terminal -- bash -c "./servidor_udp; echo 'Presionar ENTER para cerrar...'; read"

In [ ]:
gnome-terminal -- bash -c "./cliente_udp; echo 'Presionar ENTER para cerrar...'; read"

---
## A-3) Chat con identificación de clientes

La mejora respecto al chat anterior es que el servidor muestra qué cliente envió cada mensaje usando su **nombre + IP + puerto**. El servidor concurrente ya recibía el nombre, pero ahora **el mensaje reenviado a todos los otros clientes también incluye el nombre del emisor**, para que puedan identificar quién habla.


In [ ]:
cat > servidor_chat.c << 'EOF'
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <unistd.h>
#include <netinet/in.h>
#include <arpa/inet.h>
#include <sys/select.h>

#define PORT 8082
#define MAX_CLIENTS 10
#define MAX 1024

// Guardamos nombre e IP de cada cliente para identificarlo
typedef struct {
    int fd;
    char nombre[50];
    char ip[INET_ADDRSTRLEN];
    int puerto;
} Cliente;

Cliente clientes[MAX_CLIENTS];
int num_clientes = 0;

// Envía un mensaje a todos los clientes excepto al emisor
void broadcast(int emisor_fd, const char *msg) {
    for (int i = 0; i < num_clientes; i++) {
        if (clientes[i].fd != emisor_fd) {
            send(clientes[i].fd, msg, strlen(msg), 0);
        }
    }
}

int main() {
    int server_fd, new_fd;
    struct sockaddr_in server_addr, client_addr;
    socklen_t addr_len = sizeof(client_addr);
    fd_set read_fds;
    char buffer[MAX], msg[MAX + 60];

    server_fd = socket(AF_INET, SOCK_STREAM, 0);
    server_addr.sin_family = AF_INET;
    server_addr.sin_addr.s_addr = INADDR_ANY;
    server_addr.sin_port = htons(PORT);
    bind(server_fd, (struct sockaddr*)&server_addr, sizeof(server_addr));
    listen(server_fd, 5);

    printf("Servidor de chat en puerto %d...\n", PORT);

    while (1) {
        FD_ZERO(&read_fds);
        FD_SET(server_fd, &read_fds);
        int max_fd = server_fd;

        for (int i = 0; i < num_clientes; i++) {
            FD_SET(clientes[i].fd, &read_fds);
            if (clientes[i].fd > max_fd) max_fd = clientes[i].fd;
        }

        select(max_fd + 1, &read_fds, NULL, NULL, NULL);

        // Nueva conexión
        if (FD_ISSET(server_fd, &read_fds)) {
            new_fd = accept(server_fd, (struct sockaddr*)&client_addr, &addr_len);
            Cliente c;
            c.fd = new_fd;
            c.puerto = ntohs(client_addr.sin_port);
            inet_ntop(AF_INET, &client_addr.sin_addr, c.ip, INET_ADDRSTRLEN);

            // El primer mensaje del cliente es su nombre
            recv(new_fd, c.nombre, sizeof(c.nombre), 0);
            clientes[num_clientes++] = c;

            printf("[+] %s (%s:%d) se conectó\n", c.nombre, c.ip, c.puerto);
            snprintf(msg, sizeof(msg), "*** %s se unió al chat ***\n", c.nombre);
            broadcast(new_fd, msg);
        }

        // Mensaje de un cliente
        for (int i = 0; i < num_clientes; i++) {
            if (FD_ISSET(clientes[i].fd, &read_fds)) {
                memset(buffer, 0, MAX);
                int bytes = recv(clientes[i].fd, buffer, MAX, 0);

                if (bytes <= 0) {
                    printf("[-] %s se desconectó\n", clientes[i].nombre);
                    snprintf(msg, sizeof(msg), "*** %s salió del chat ***\n", clientes[i].nombre);
                    broadcast(clientes[i].fd, msg);
                    close(clientes[i].fd);
                    // Eliminar del array
                    clientes[i] = clientes[--num_clientes];
                } else {
                    // Reenviar el mensaje con el nombre del emisor
                    snprintf(msg, sizeof(msg), "[%s]: %s", clientes[i].nombre, buffer);
                    printf("%s", msg);
                    broadcast(clientes[i].fd, msg);
                }
            }
        }
    }

    close(server_fd);
    return 0;
}
EOF
gcc servidor_chat.c -o servidor_chat && echo "Compilado OK"

El cliente del chat es el mismo `cliente_select` del ejemplo anterior (conectado al puerto 8082), pero ahora lo modificamos para que envíe el nombre al conectarse:


In [ ]:
cat > cliente_chat.c << 'EOF'
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <unistd.h>
#include <arpa/inet.h>
#include <sys/select.h>

#define PORT 8082
#define MAX 1024

int main() {
    int sock;
    struct sockaddr_in server_addr;
    char buffer[MAX], nombre[50];
    fd_set read_fds;

    sock = socket(AF_INET, SOCK_STREAM, 0);
    server_addr.sin_family = AF_INET;
    server_addr.sin_port = htons(PORT);
    inet_pton(AF_INET, "127.0.0.1", &server_addr.sin_addr);
    connect(sock, (struct sockaddr*)&server_addr, sizeof(server_addr));

    // Enviar nombre al conectarse (el servidor lo usa para identificar)
    printf("Tu nombre: ");
    fgets(nombre, sizeof(nombre), stdin);
    nombre[strcspn(nombre, "\n")] = 0;
    send(sock, nombre, strlen(nombre), 0);

    printf("Conectado al chat como '%s'. Escribí mensajes:\n", nombre);

    while (1) {
        FD_ZERO(&read_fds);
        FD_SET(0, &read_fds);
        FD_SET(sock, &read_fds);
        select(sock + 1, &read_fds, NULL, NULL, NULL);

        if (FD_ISSET(0, &read_fds)) {
            fgets(buffer, MAX, stdin);
            send(sock, buffer, strlen(buffer), 0);
        }

        if (FD_ISSET(sock, &read_fds)) {
            int bytes = recv(sock, buffer, MAX - 1, 0);
            if (bytes <= 0) { printf("Servidor cerró la conexión.\n"); break; }
            buffer[bytes] = '\0';
            printf("%s", buffer);
        }
    }

    close(sock);
    return 0;
}
EOF
gcc cliente_chat.c -o cliente_chat && echo "Compilado OK"

In [ ]:
gnome-terminal -- bash -c "./servidor_chat; echo 'Presionar ENTER para cerrar...'; read"

In [ ]:
# Abrir dos clientes para probar el chat entre ellos
gnome-terminal -- bash -c "./cliente_chat; echo 'Presionar ENTER para cerrar...'; read"
gnome-terminal -- bash -c "./cliente_chat; echo 'Presionar ENTER para cerrar...'; read"

---
## A-4) Clientes automáticos para observar el comportamiento de fork()

La idea es lanzar **múltiples clientes al mismo tiempo** que envían mensajes automáticamente (sin intervención humana). Así podemos ver en el servidor concurrente cómo se crean varios procesos hijo con `fork()`, cada uno con su propio PID atendiendo a un cliente distinto de forma simultánea.

El cliente automático se conecta, envía N mensajes con una pequeña pausa entre ellos, y luego cierra la conexión.


In [ ]:
cat > cliente_automatico.c << 'EOF'
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <unistd.h>
#include <netinet/in.h>
#include <arpa/inet.h>

#define PORT 8080
#define MAX 1024

int main(int argc, char *argv[]) {
    // Recibe el nombre como argumento para identificar cada instancia
    char *nombre = (argc > 1) ? argv[1] : "ClienteAuto";

    int sockfd;
    struct sockaddr_in server_addr;
    char buffer[MAX];

    sockfd = socket(AF_INET, SOCK_STREAM, 0);
    server_addr.sin_family = AF_INET;
    server_addr.sin_port = htons(PORT);
    inet_pton(AF_INET, "127.0.0.1", &server_addr.sin_addr);

    if (connect(sockfd, (struct sockaddr*)&server_addr, sizeof(server_addr)) < 0) {
        perror("connect");
        return 1;
    }

    // Enviar nombre al servidor
    send(sockfd, nombre, strlen(nombre), 0);

    // Enviar 5 mensajes automáticos con pausa entre ellos
    for (int i = 1; i <= 5; i++) {
        snprintf(buffer, MAX, "Mensaje %d de %s\n", i, nombre);
        send(sockfd, buffer, strlen(buffer), 0);
        printf("[%s] Enviado: %s", nombre, buffer);

        // Recibir respuesta del servidor
        memset(buffer, 0, MAX);
        int bytes = recv(sockfd, buffer, MAX, 0);
        if (bytes > 0) printf("[%s] Respuesta: %s", nombre, buffer);

        sleep(1); // pausa para que se note la concurrencia
    }

    // Finalizar
    send(sockfd, "FIN\n", 4, 0);
    close(sockfd);
    printf("[%s] Conexión cerrada.\n", nombre);
    return 0;
}
EOF
gcc cliente_automatico.c -o cliente_automatico && echo "Compilado OK"

Primero iniciar el **servidor concurrente** (el ya compilado `servidor_tcp_concurrente_bash`). Luego ejecutar esta celda para lanzar 3 clientes simultáneamente en terminales separadas y observar cómo el servidor crea un proceso hijo (`fork()`) para cada uno:


In [ ]:
# Lanzar el servidor concurrente
gnome-terminal -- bash -c "./servidor_tcp_concurrente_bash; echo 'Presionar ENTER para cerrar...'; read"

In [ ]:
# Lanzar 3 clientes automáticos en paralelo, cada uno en su propia terminal
# En el servidor se verá cómo se crean 3 procesos hijos con fork() distintos
gnome-terminal -- bash -c "./cliente_automatico ClienteA; echo 'Presionar ENTER...'; read"
gnome-terminal -- bash -c "./cliente_automatico ClienteB; echo 'Presionar ENTER...'; read"
gnome-terminal -- bash -c "./cliente_automatico ClienteC; echo 'Presionar ENTER...'; read"

**Qué observar en el servidor:** cada vez que llega un cliente nuevo, aparece una línea como:
```
[PID 1234] Cliente conectado. Proceso hijo PID 1350
[PID 1234] Cliente conectado. Proceso hijo PID 1351
[PID 1234] Cliente conectado. Proceso hijo PID 1352
```
El proceso padre (PID 1234) sigue aceptando conexiones mientras los hijos (PIDs distintos) atienden a cada cliente en paralelo. Los mensajes se intercalan en la salida porque los tres hijos corren **simultáneamente**.


## B) Servidores con concurrencia aparente usando select()

¿Qué hace select()?
select() permite monitorear múltiples sockets (conexiones de red), archivos, o entradas estándar (stdin) y saber cuál de ellos está listo para lectura, escritura o tiene errores. Se usa en servidores que manejan múltiples clientes al mismo tiempo. Permite manejar múltiples clientes sin necesidad de hilos (threads) o procesos hijos.

## Cliente con select()

In [ ]:
echo '
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <unistd.h>
#include <errno.h>
#include <sys/socket.h>
#include <sys/types.h>
#include <arpa/inet.h>
#include <netinet/in.h>
#include <sys/select.h>

#define PORT 12345
#define MAX_CLIENTS 10
#define BUFFER_SIZE 1024

int main() {
    int server_fd, new_socket, client_sockets[MAX_CLIENTS];
    struct sockaddr_in address;
    fd_set read_fds;
    int max_sd, sd, activity, valread;
    char buffer[BUFFER_SIZE];

    // CORRECCIÓN: en el código original addrlen se declaraba dentro del bloque
    // if (FD_ISSET(server_fd, &read_fds)) como:
    //     socklen_t addrlen = sizeof(address);
    // Eso la dejaba fuera de scope al usarse más abajo en getpeername() dentro
    // del for de clientes, causando un error de compilación ("undeclared identifier").
    // Solución: declararla aquí al inicio de main() y solo asignarle el valor
    // dentro del if, sin redeclararla.
    socklen_t addrlen;

    // Inicializar todos los clientes a 0
    for (int i = 0; i < MAX_CLIENTS; i++) {
        client_sockets[i] = 0;
    }

    // Crear socket
    if ((server_fd = socket(AF_INET, SOCK_STREAM, 0)) == 0) {
        perror("socket falló");
        exit(EXIT_FAILURE);
    }

    // Configurar dirección del servidor
    address.sin_family = AF_INET;
    address.sin_addr.s_addr = INADDR_ANY;
    address.sin_port = htons(PORT);

    // Enlazar socket al puerto
    if (bind(server_fd, (struct sockaddr *)&address, sizeof(address)) < 0) {
        perror("bind falló");
        exit(EXIT_FAILURE);
    }

    // Escuchar conexiones entrantes
    if (listen(server_fd, 3) < 0) {
        perror("listen");
        exit(EXIT_FAILURE);
    }

    printf("Servidor escuchando en puerto %d...\n", PORT);

    while (1) {
        // Limpiar conjunto de descriptores
        FD_ZERO(&read_fds);
        FD_SET(server_fd, &read_fds);
        max_sd = server_fd;

        // Agregar sockets de clientes al conjunto
        for (int i = 0; i < MAX_CLIENTS; i++) {
            sd = client_sockets[i];
            if (sd > 0) FD_SET(sd, &read_fds);
            if (sd > max_sd) max_sd = sd;
        }

        // Esperar actividad
        activity = select(max_sd + 1, &read_fds, NULL, NULL, NULL);

        if ((activity < 0) && (errno != EINTR)) {
            printf("Error en select\n");
        }

        // Nueva conexión
        if (FD_ISSET(server_fd, &read_fds)) {
            // Solo se asigna el valor, no se redeclara (ver comentario al inicio de main)
            addrlen = sizeof(address);
            if ((new_socket = accept(server_fd, (struct sockaddr *)&address, &addrlen)) < 0) {
                perror("accept");
                exit(EXIT_FAILURE);
            }

            printf("Nueva conexión: socket fd %d, IP %s, puerto %d\n",
                   new_socket, inet_ntoa(address.sin_addr), ntohs(address.sin_port));

            // Agregar a la lista de clientes
            for (int i = 0; i < MAX_CLIENTS; i++) {
                if (client_sockets[i] == 0) {
                    client_sockets[i] = new_socket;
                    break;
                }
            }
        }

        // Revisar mensajes de clientes
        for (int i = 0; i < MAX_CLIENTS; i++) {
            sd = client_sockets[i];

            if (FD_ISSET(sd, &read_fds)) {
                if ((valread = read(sd, buffer, BUFFER_SIZE)) == 0) {
                    // Desconexión: addrlen es visible aquí gracias a la corrección
                    getpeername(sd, (struct sockaddr*)&address, (socklen_t*)&addrlen);
                    printf("Cliente desconectado: IP %s, puerto %d\n",
                           inet_ntoa(address.sin_addr), ntohs(address.sin_port));
                    close(sd);
                    client_sockets[i] = 0;
                } else {
                    buffer[valread] = '\0';
                    printf("Mensaje recibido: %s", buffer);

                    // Reenviar mensaje a otros clientes
                    for (int j = 0; j < MAX_CLIENTS; j++) {
                        if (client_sockets[j] != 0 && client_sockets[j] != sd) {
                            send(client_sockets[j], buffer, strlen(buffer), 0);
                        }
                    }
                }
            }
        }
    }

    return 0;
}' > servidor_select.c

Compilo el código usando Bash C y verifico que el ejecutable esté disponible

In [ ]:
gcc servidor_select.c -o servidor_select | ls

Si abro la ventana terminal de esta forma se cerrará al finalizar la ejecución del servidor:

In [ ]:
gnome-terminal -- bash -c "./servidor_select; echo 'Presionar ENTER para cerrar...'; read"

## Cliente con select()

In [ ]:
echo '#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <unistd.h>
#include <arpa/inet.h>
#include <sys/socket.h>
#include <sys/select.h>

#define PORT 12345
#define BUFFER_SIZE 1024

int main() {
    int sock;
    struct sockaddr_in server_addr;
    char buffer[BUFFER_SIZE];
    fd_set read_fds;

    // Crear socket
    sock = socket(AF_INET, SOCK_STREAM, 0);
    if (sock < 0) {
        perror("socket");
        exit(EXIT_FAILURE);
    }

    // Configurar dirección del servidor
    server_addr.sin_family = AF_INET;
    server_addr.sin_port = htons(PORT);
    inet_pton(AF_INET, "127.0.0.1", &server_addr.sin_addr);

    // Conectarse al servidor
    if (connect(sock, (struct sockaddr *)&server_addr, sizeof(server_addr)) < 0) {
        perror("connect");
        exit(EXIT_FAILURE);
    }

    printf("Conectado al servidor. Puedes escribir mensajes:\n");

    while (1) {
        FD_ZERO(&read_fds);
        FD_SET(0, &read_fds);      // Entrada estándar
        FD_SET(sock, &read_fds);   // Socket

        if (select(sock + 1, &read_fds, NULL, NULL, NULL) < 0) {
            perror("select");
            exit(EXIT_FAILURE);
        }

        // Entrada del usuario
        if (FD_ISSET(0, &read_fds)) {
            fgets(buffer, BUFFER_SIZE, stdin);
            send(sock, buffer, strlen(buffer), 0);
        }

        // Mensaje del servidor
        if (FD_ISSET(sock, &read_fds)) {
            int valread = read(sock, buffer, BUFFER_SIZE - 1);
            if (valread <= 0) {
                printf("Desconectado del servidor.\n");
                break;
            }
            buffer[valread] = '\0';
            printf("Mensaje: %s", buffer);
        }
    }

    close(sock);
    return 0;
}' > cliente_select.c

Compilo el código usando Bash C y verifico que el ejecutable esté disponible:

In [ ]:
gcc cliente_select.c -o cliente_select | ls

Si abro la ventana terminal de esta forma se cerrará al finalizar la ejecución del servidor:

In [ ]:
gnome-terminal -- bash -c "./cliente_select; echo 'Presionar ENTER para cerrar...'; read"

### SERVIDOR PASO A PASO:
(1) Declaraciones y configuraciones iniciales

- #define PORT 12345
- #define MAX_CLIENTS 10
- #define BUFFER_SIZE 1024

- Puerto donde el servidor va a escuchar.
- Máximo de clientes simultáneos.
- Tamaño del buffer de mensajes.

int server_fd, new_socket, client_sockets[MAX_CLIENTS];
struct sockaddr_in address;
fd_set read_fds;

- server_fd: socket del servidor (para aceptar conexiones).
- new_socket: se usa cuando un cliente se conecta.
- client_sockets[]: array para guardar los sockets de los clientes.
- fd_set read_fds: conjunto de descriptores de archivo para pasar a select().

(2) Crear el socket

server_fd = socket(AF_INET, SOCK_STREAM, 0);

- Crea un socket TCP.

(3) Asignar dirección y puerto

address.sin_family = AF_INET;
address.sin_addr.s_addr = INADDR_ANY;
address.sin_port = htons(PORT);

- AF_INET: IPv4.
- INADDR_ANY: Acepta conexiones en cualquier interfaz de red.
- htons: convierte el número de puerto a formato de red.

(4) Enlazar y escuchar

bind(server_fd, (struct sockaddr *)&address, sizeof(address));
listen(server_fd, 3);

- Asocia el socket a una dirección IP y puerto.
- Empieza a escuchar conexiones entrantes.

(5) Bucle principal

while (1) {
    FD_ZERO(&read_fds); // limpiar conjunto
    FD_SET(server_fd, &read_fds); // añadir el socket servidor

- Limpiamos el conjunto de descriptores (FD_ZERO).
- Añadimos el descriptor del servidor (para detectar nuevas conexiones).

    for (int i = 0; i < MAX_CLIENTS; i++) {
        sd = client_sockets[i];
        if (sd > 0) FD_SET(sd, &read_fds);
    }

- Añadimos cada socket de cliente activo al conjunto read_fds.

    activity = select(max_sd + 1, &read_fds, NULL, NULL, NULL);

- select() espera hasta que alguno de los sockets esté listo para lectura.

(6) Si hay una nueva conexión

if (FD_ISSET(server_fd, &read_fds)) {
    new_socket = accept(...);
    client_sockets[i] = new_socket;
}

- Se acepta la nueva conexión.
- Se guarda el nuevo socket en el array client_sockets.

(7) Si un cliente envía un mensaje

if (FD_ISSET(sd, &read_fds)) {
    valread = read(sd, buffer, BUFFER_SIZE);
}

- Si read() devuelve 0, el cliente cerró la conexión.
- Si devuelve > 0, recibimos un mensaje y lo reenviamos a los demás clientes:

send(client_sockets[j], buffer, strlen(buffer), 0);

### CLIENTE PASO A PASO:
(1) Crear socket

sock = socket(AF_INET, SOCK_STREAM, 0);

- Creamos un socket TCP.

(2) Conectar al servidor

inet_pton(AF_INET, "127.0.0.1", &server_addr.sin_addr);
connect(sock, (struct sockaddr *)&server_addr, sizeof(server_addr));

- Convertimos la IP en texto a binario (inet_pton).
- Conectamos al socket del servidor.

(3) Bucle principal con select()

FD_SET(0, &read_fds);     // entrada estándar (teclado)
FD_SET(sock, &read_fds);  // socket de red
select(sock + 1, &read_fds, NULL, NULL, NULL);

- Monitorea si hay datos listos para leer desde:
- El teclado (stdin)
- El servidor (sock)

(4) Si el usuario escribe un mensaje

fgets(buffer, BUFFER_SIZE, stdin);
send(sock, buffer, strlen(buffer), 0);

- Leemos desde teclado y enviamos al servidor.

(5) Si el servidor envía un mensaje

valread = read(sock, buffer, BUFFER_SIZE - 1);

- Recibimos y mostramos el mensaje en pantalla.

### SERVIDOR
Notas importantes

- #include <sys/select.h> // select(), fd_set, FD_SET, etc.
- #include <sys/socket.h> // socket(), bind(), listen(), etc.
- #include <netinet/in.h> // struct sockaddr_in
- int server_fd = socket(AF_INET, SOCK_STREAM, 0);


FD_ZERO(&read_fds);              // Limpia el conjunto
FD_SET(server_fd, &read_fds);    // Agrega el socket del servidor
max_sd = server_fd;

- FD_ZERO(): limpia el conjunto read_fds.
- FD_SET(server_fd, &read_fds): agrega el descriptor del socket servidor.
- max_sd: guarda el descriptor más alto. Es requerido por select().

Agregar los sockets de clientes activos

for (int i = 0; i < MAX_CLIENTS; i++) {
    sd = client_sockets[i];
    if (sd > 0)
        FD_SET(sd, &read_fds);
    if (sd > max_sd)
        max_sd = sd;
}

- Recorre el array de sockets.
- Agrega los clientes activos a read_fds.
- Actualiza max_sd si el descriptor es mayor.

Llamar a select()

activity = select(max_sd + 1, &read_fds, NULL, NULL, NULL);

- Espera hasta que uno o más sockets estén listos para lectura.
- max_sd + 1: límite superior de descriptores a monitorear.
- Los otros parámetros son NULL: no monitorea escritura ni errores.

Nueva conexión entrante

if (FD_ISSET(server_fd, &read_fds)) {
    new_socket = accept(server_fd, ...);
}

- FD_ISSET() verifica si el servidor tiene una nueva conexión.
- accept() acepta la conexión y retorna un nuevo descriptor para el cliente.

Guardar el nuevo cliente

for (int i = 0; i < MAX_CLIENTS; i++) {
    if (client_sockets[i] == 0) {
        client_sockets[i] = new_socket;
        break;
    }
}

- Se guarda el socket del nuevo cliente en el array.

Verificar si un cliente envió un mensaje

if (FD_ISSET(sd, &read_fds)) {
    valread = read(sd, buffer, BUFFER_SIZE);
}

- FD_ISSET() verifica si el cliente ha enviado algo.
- read() lee los datos del socket.

Si el cliente cerró conexión

if (valread == 0) {
    close(sd);
    client_sockets[i] = 0;
}

- read() devuelve 0 si el cliente se desconectó.
- Se cierra el socket y se borra del array.

Reenviar mensaje a otros clientes

for (int j = 0; j < MAX_CLIENTS; j++) {
    if (client_sockets[j] != 0 && client_sockets[j] != sd)
        send(client_sockets[j], buffer, strlen(buffer), 0);
}

- Se envía el mensaje recibido a todos los clientes excepto al que lo envió.

## CLIENTE

Bucle de comunicación

while (1) {
    FD_ZERO(&read_fds);
    FD_SET(0, &read_fds);    // Entrada por teclado
    FD_SET(sock, &read_fds); // Socket con el servidor
    select(sock + 1, &read_fds, NULL, NULL, NULL);

- FD_SET(0): monitorea teclado (stdin) por si el usuario escribe.
- FD_SET(sock): monitorea el socket con el servidor.
- select() espera a que uno de ellos tenga datos para leer.

Leer del teclado y enviar al servidor

if (FD_ISSET(0, &read_fds)) {
    fgets(buffer, BUFFER_SIZE, stdin);
    send(sock, buffer, strlen(buffer), 0);
}

- Si el usuario escribió algo, lo enviamos al servidor con send().

Leer mensaje del servidor

if (FD_ISSET(sock, &read_fds)) {
    valread = read(sock, buffer, BUFFER_SIZE - 1);
    if (valread <= 0) break;
    buffer[valread] = '\0';
    printf("Mensaje: %s", buffer);
}

- Si el servidor envió algo, se muestra en pantalla.
- Si read() devuelve 0 o menor, el servidor cerró la conexión.

¿Por qué usamos FD_SET, FD_ZERO, FD_ISSET?

- FD_ZERO(&set): limpia el conjunto.
- FD_SET(fd, &set): agrega el descriptor fd al conjunto.
- FD_ISSET(fd, &set): verifica si fd tiene datos listos.

select() usa estos conjuntos para saber qué descriptor está listo para operar.

El uso de FD_SET() en el cliente es necesario para poder usar select() y detectar cuándo el usuario escribe algo (stdin) o cuándo llega un mensaje del servidor sin bloquearte en una sola operación.

En el cliente tenemos dos posibles fuentes de entrada:

- Teclado del usuario (stdin, que es descriptor 0)
- Socket de conexión al servidor (por ejemplo, descriptor 4)

Para saber cuál de esas fuentes tiene datos disponibles para leer, usamos select() y le pasamos un fd_set con esos dos descriptores:

FD_ZERO(&read_fds);         // Limpia el conjunto
FD_SET(0, &read_fds);       // Agrega teclado (stdin)
FD_SET(sock, &read_fds);    // Agrega socket con el servidor
select(sock + 1, &read_fds, NULL, NULL, NULL);

¿Y si no uso FD_SET(0, &read_fds)?

- El cliente no podría detectar si el usuario escribió algo.
- Solo sabría si el servidor envió un mensaje.
- Habría que usar fgets() o scanf() que bloquean el programa, y no se podría recibir mensajes mientras el usuario no escriba.

Ventajas de usar select() en el cliente con FD_SET()

- Permite que el cliente sea reactivo y no se bloquee.
- Permite recibir mensajes mientras el usuario escribe.
- Permite enviar mensajes apenas el usuario los escriba.
- Código más profesional.



## Cliente sin select() (versión simplificada y bloqueante)

In [ ]:
echo '#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <unistd.h>
#include <arpa/inet.h>

#define PORT 12345
#define BUFFER_SIZE 1024

int main() {
    int sock;
    struct sockaddr_in server_addr;
    char buffer[BUFFER_SIZE];

    // 1. Crear socket
    sock = socket(AF_INET, SOCK_STREAM, 0);
    if (sock == -1) {
        perror("Error al abrir el socket ...");
        exit(EXIT_FAILURE);
    }

    // 2. Configurar dirección del servidor
    server_addr.sin_family = AF_INET;
    server_addr.sin_port = htons(PORT);
    server_addr.sin_addr.s_addr = inet_addr("127.0.0.1");

    // 3. Conectar al servidor
    if (connect(sock, (struct sockaddr *)&server_addr, sizeof(server_addr)) < 0) {
        perror("Error al conectar");
        exit(EXIT_FAILURE);
    }

    printf("Conectado al servidor. Escribe mensajes y presiona Enter.\n");

    // 4. Bucle principal bloqueante
    while (1) {
        printf("> ");
        fgets(buffer, BUFFER_SIZE, stdin);  // BLOQUEA esperando entrada
        send(sock, buffer, strlen(buffer), 0);

        // Leer respuesta del servidor
        int valread = read(sock, buffer, BUFFER_SIZE - 1);
        if (valread <= 0) {
            printf("Conexión cerrada por el servidor.\n");
            break;
        }

        buffer[valread] = '\0';
        printf("Servidor: %s", buffer);
    }

    close(sock);
    return 0;
}' > cliente_bloqueante.c

Compilar y abrir termina según se explicó anteriormente

**Ejercicios a realizar durante la clase. Incluir los códigos dentro de este mismo Notebook**

B-1) Comparar cliente con select y bloqueante.   
B-2) Modificar el cliente bloqueante para que sea no bloqueante.   
B-3) Compara cliente con select y no bloqueante sin select.   
B-4) Conclusiones.   

---
## B-1) Comparación: cliente con select() vs cliente bloqueante

| Característica | Cliente bloqueante | Cliente con select() |
|---|---|---|
| `fgets()` bloquea el programa | Sí. Mientras espera que el usuario escriba, **no puede recibir mensajes** | No. `select()` detecta cuál de los dos (teclado o socket) tiene datos |
| ¿Puede recibir mensajes mientras escribe? | No | Sí |
| Complejidad del código | Simple | Un poco más complejo |
| Uso en producción | No recomendado para chat | Correcto para manejar múltiples fuentes de entrada |

**Conclusión B-1:** El cliente bloqueante queda "atascado" en `fgets()` y no puede ver mensajes entrantes hasta que el usuario presione Enter. El cliente con `select()` monitorea ambas fuentes simultáneamente y reacciona a la que tenga datos primero.


---
## B-2) Cliente bloqueante convertido a no bloqueante

Un socket **no bloqueante** hace que las llamadas como `recv()` y `send()` retornen **inmediatamente** si no hay datos disponibles, en lugar de esperar. Se activa con `fcntl()` usando la flag `O_NONBLOCK`.

Cuando `recv()` no tiene datos, retorna `-1` y `errno == EAGAIN` (en lugar de bloquearse). Así el programa puede seguir ejecutándose y verificar el socket en cada vuelta del bucle.


In [ ]:
cat > cliente_no_bloqueante.c << 'EOF'
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <unistd.h>
#include <errno.h>
#include <fcntl.h>
#include <arpa/inet.h>

#define PORT 12345
#define BUFFER_SIZE 1024

int main() {
    int sock;
    struct sockaddr_in server_addr;
    char buffer[BUFFER_SIZE];

    sock = socket(AF_INET, SOCK_STREAM, 0);
    server_addr.sin_family = AF_INET;
    server_addr.sin_port = htons(PORT);
    server_addr.sin_addr.s_addr = inet_addr("127.0.0.1");
    connect(sock, (struct sockaddr*)&server_addr, sizeof(server_addr));

    // Poner el socket en modo NO BLOQUEANTE
    // fcntl obtiene los flags actuales y le agrega O_NONBLOCK
    int flags = fcntl(sock, F_GETFL, 0);
    fcntl(sock, F_SETFL, flags | O_NONBLOCK);

    // También poner stdin en modo no bloqueante
    flags = fcntl(0, F_GETFL, 0);
    fcntl(0, F_SETFL, flags | O_NONBLOCK);

    printf("Conectado (modo no bloqueante). Escribí mensajes:\n");

    while (1) {
        // Intentar leer del teclado (no bloquea)
        memset(buffer, 0, BUFFER_SIZE);
        int leido = read(0, buffer, BUFFER_SIZE - 1);
        if (leido > 0) {
            send(sock, buffer, leido, 0);
        }

        // Intentar leer del servidor (no bloquea)
        memset(buffer, 0, BUFFER_SIZE);
        int bytes = recv(sock, buffer, BUFFER_SIZE - 1, 0);
        if (bytes > 0) {
            buffer[bytes] = '\0';
            printf("Servidor: %s", buffer);
        } else if (bytes == 0) {
            printf("Servidor cerró la conexión.\n");
            break;
        }
        // Si bytes == -1 y errno == EAGAIN: no hay datos aún, continuar el bucle

        usleep(10000); // pequeña pausa para no consumir 100% de CPU
    }

    close(sock);
    return 0;
}
EOF
gcc cliente_no_bloqueante.c -o cliente_no_bloqueante && echo "Compilado OK"

In [ ]:
gnome-terminal -- bash -c "./cliente_no_bloqueante; echo 'Presionar ENTER para cerrar...'; read"

---
## B-3) Comparación: cliente con select() vs cliente no bloqueante sin select()

Ambos logran que el cliente **no se bloquee**, pero de manera diferente:

| Característica | Cliente con select() | Cliente no bloqueante (sin select) |
|---|---|---|
| Mecanismo | `select()` espera pasivamente hasta que haya actividad | Bucle activo que pregunta continuamente ("polling") |
| Uso de CPU | Bajo: el proceso duerme hasta que hay datos | Alto: el bucle corre constantemente aunque no haya datos |
| Complejidad | Hay que armar y recargar el `fd_set` en cada iteración | Más simple de entender, solo se agregan flags con `fcntl()` |
| Escalabilidad | Excelente: puede monitorear muchos fds a la vez | Mala: cada socket extra requiere más código manual |
| `errno == EAGAIN` | No hace falta manejarlo (select lo evita) | Hay que verificarlo explícitamente en cada `recv()` |

**Conclusión B-3:** `select()` es el enfoque correcto para producción. El no bloqueante sin `select()` es conceptualmente simple pero desperdicia CPU haciendo polling constante.

---
## B-4) Conclusiones generales de la sección B

1. **Bloqueante:** más simple de escribir, pero inutilizable en aplicaciones donde hay múltiples fuentes de entrada simultáneas (teclado + red). El programa queda "congelado" esperando una sola operación.

2. **No bloqueante con `fcntl(O_NONBLOCK)`:** el socket nunca congela el programa. Pero sin algún mecanismo de espera inteligente, el bucle consume CPU innecesariamente.

3. **`select()` con sockets bloqueantes:** la mejor opción del grupo. El proceso duerme hasta que alguna fuente tenga datos, reacciona solo cuando es necesario, y el código es claro respecto a qué se está monitoreando.

4. **Regla práctica:** para manejar una sola conexión simple, bloqueante alcanza. Para múltiples conexiones o múltiples fuentes de entrada, usar `select()` (o equivalentes modernos como `poll()` o `epoll()`).


## Consigna del TP 1

- Explicar qué es un socket y los diferentes tipos de sockets.
- Cuáles son las estructuras necesarias para operar con sockets en el modelo C-S y cómo se hace para ingresar los datos requeridos.
-- Explicar con un ejemplo.
- Explicar modo bloqueante y no bloqueante en sockets y cuáles son los “sockets calls” a los cuales se pueden aplicar estos modos.
-- Explicar las funciones necesarias.
- Describir el modelo C-S aplicado a un servidor con Concurrencia Real.
- Escribir un ejemplo en lenguaje C distinto del ya presentado.
- Cómo se cierra una conexión C-S ?. Métodos que eviten la pérdida de informacion.
- Probar el código del chat entre cliente y servidor. Cambiar tipo de socket y volver a probar.
- Presentar las respuestas en archivos Jupyter Notebook o PDF.

C-S = Cliente - Servidor  
- Bibliografía a consultar:  
-- Clases Teóricas.  
-- Brian "Beej Jorgensen" Hall. Beej's Guide to Network Programming: Using Internet Sockets. 2019.  
-- Douglas E. Comer, David L. Stevens. Internetworking With Tcp/Ip: Client-Server Programming and Applications. Prentice Hall. 2011.  

---
## 1) ¿Qué es un socket y cuáles son sus tipos?

Un **socket** es un punto extremo de una comunicación entre dos procesos, ya sea en la misma máquina o en máquinas distintas a través de una red. Actúa como una interfaz entre la aplicación y la capa de transporte (TCP/UDP).

### Tipos de sockets:

- **SOCK_STREAM (TCP):** orientado a conexión. Garantiza orden de llegada, sin duplicados, y con confirmación de entrega. Requiere `connect()` / `accept()`. Ideal para transferencia de datos confiable.

- **SOCK_DGRAM (UDP):** sin conexión. Envía datagramas independientes. No garantiza orden ni entrega. Más rápido y liviano. Ideal para streaming, juegos en red, DNS.

- **SOCK_RAW:** acceso directo a las cabeceras del protocolo (IP, ICMP). Usado para herramientas de diagnóstico como `ping` o sniffers.

- **SOCK_SEQPACKET:** similar a STREAM pero preserva los límites de los mensajes (menos común).

La familia de dirección más usada es `AF_INET` (IPv4). También existe `AF_INET6` (IPv6) y `AF_UNIX` (comunicación local entre procesos).


---
## 2) Estructuras necesarias para operar con sockets en el modelo C-S

La estructura principal es `sockaddr_in`, que representa una dirección de red IPv4:

```c
struct sockaddr_in {
    sa_family_t    sin_family;   // Familia: AF_INET
    in_port_t      sin_port;     // Puerto (en formato de red, big-endian)
    struct in_addr sin_addr;     // Dirección IP
    char           sin_zero[8];  // Relleno (se pone a 0)
};
```

### Funciones para cargar los datos:

- `htons(puerto)` — convierte el puerto de formato local a formato de red (big-endian).
- `inet_pton(AF_INET, "192.168.1.1", &addr.sin_addr)` — convierte una IP en texto a binario.
- `INADDR_ANY` — en el servidor, acepta conexiones en cualquier interfaz de red.

### Ejemplo completo de configuración:

```c
struct sockaddr_in server_addr;

server_addr.sin_family = AF_INET;              // IPv4
server_addr.sin_port   = htons(8080);          // Puerto 8080
inet_pton(AF_INET, "127.0.0.1",               // IP destino (cliente)
          &server_addr.sin_addr);
memset(server_addr.sin_zero, 0, 8);            // limpiar relleno
```

En el **servidor** se usa `INADDR_ANY` en vez de una IP específica:
```c
server_addr.sin_addr.s_addr = INADDR_ANY;      // acepta en todas las interfaces
```


---
## 3) Modo bloqueante y no bloqueante en sockets

### Modo bloqueante (por defecto)
Cuando se llama a una función como `recv()`, el proceso **se suspende** hasta que haya datos disponibles. No consume CPU mientras espera, pero no puede hacer nada más.

### Modo no bloqueante
Se activa con `fcntl()` usando `O_NONBLOCK`. Las llamadas retornan **inmediatamente** aunque no haya datos. Si no hay nada disponible, retornan `-1` y `errno == EAGAIN`.

```c
int flags = fcntl(sock, F_GETFL, 0);
fcntl(sock, F_SETFL, flags | O_NONBLOCK);
```

### Socket calls a los que aplica el modo bloqueante/no bloqueante:

| Llamada | Bloquea cuando... |
|---|---|
| `accept()` | No hay ningún cliente conectado |
| `connect()` | La conexión no se estableció aún |
| `recv()` / `read()` | No hay datos para leer |
| `send()` / `write()` | El buffer de envío está lleno |
| `recvfrom()` | No hay datagramas disponibles (UDP) |

### `select()` como alternativa inteligente
En lugar de usar sockets no bloqueantes con polling, `select()` permite **esperar eficientemente** sobre múltiples descriptores: el proceso duerme hasta que alguno esté listo, sin desperdiciar CPU.


---
## 4) Modelo C-S con Concurrencia Real (fork)

En un servidor con **concurrencia real**, el proceso padre acepta una conexión y llama a `fork()` para crear un proceso hijo. El hijo atiende al cliente de forma independiente mientras el padre vuelve a esperar nuevas conexiones en `accept()`.

```
Padre (server_fd)
    └─ accept() ──► llega ClienteA
        └─ fork()
            ├─ Hijo 1 (PID 1350): atiende ClienteA
            └─ Padre sigue en accept()
                └─ llega ClienteB
                    └─ fork()
                        ├─ Hijo 2 (PID 1351): atiende ClienteB
                        └─ Padre sigue en accept()...
```

**Características:**
- Cada cliente tiene su propio proceso con su propio espacio de memoria.
- Verdadera ejecución en paralelo (el SO los planifica en distintos núcleos o con time-sharing).
- `signal(SIGCHLD, SIG_IGN)` evita procesos zombies cuando los hijos terminan.
- El hijo debe cerrar `server_fd` y el padre debe cerrar `client_fd` para no desperdiciar descriptores.


---
## 5) Ejemplo en C distinto al presentado: servidor de eco con múltiples clientes (usando threads)

En lugar de `fork()`, este ejemplo usa **threads POSIX (`pthread`)** para lograr concurrencia real. Cada cliente es atendido por un hilo independiente dentro del mismo proceso, compartiendo el mismo espacio de memoria.


In [ ]:
cat > servidor_threads.c << 'EOF'
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <unistd.h>
#include <netinet/in.h>
#include <arpa/inet.h>
#include <pthread.h>

#define PORT 8083
#define MAX 1024

// Cada hilo recibe el fd del cliente como argumento
void *atender_cliente(void *arg) {
    int client_fd = *(int*)arg;
    free(arg); // liberar memoria del argumento
    char buffer[MAX];

    printf("[Hilo %lu] Cliente conectado\n", pthread_self());

    while (1) {
        memset(buffer, 0, MAX);
        int bytes = recv(client_fd, buffer, MAX, 0);
        if (bytes <= 0) {
            printf("[Hilo %lu] Cliente desconectado\n", pthread_self());
            break;
        }
        printf("[Hilo %lu] Recibido: %s", pthread_self(), buffer);
        send(client_fd, buffer, bytes, 0); // eco
    }

    close(client_fd);
    return NULL;
}

int main() {
    int server_fd, *client_fd;
    struct sockaddr_in server_addr, client_addr;
    socklen_t addr_len = sizeof(client_addr);
    pthread_t tid;

    server_fd = socket(AF_INET, SOCK_STREAM, 0);
    server_addr.sin_family = AF_INET;
    server_addr.sin_addr.s_addr = INADDR_ANY;
    server_addr.sin_port = htons(PORT);
    bind(server_fd, (struct sockaddr*)&server_addr, sizeof(server_addr));
    listen(server_fd, 5);

    printf("Servidor con threads en puerto %d...\n", PORT);

    while (1) {
        // Allocar en heap para evitar race condition con la variable local
        client_fd = malloc(sizeof(int));
        *client_fd = accept(server_fd, (struct sockaddr*)&client_addr, &addr_len);

        // Crear un hilo para atender a este cliente
        // pthread_detach permite que el hilo libere sus recursos al terminar
        pthread_create(&tid, NULL, atender_cliente, client_fd);
        pthread_detach(tid);
    }

    close(server_fd);
    return 0;
}
EOF
# -lpthread: enlazar la biblioteca de threads POSIX
gcc servidor_threads.c -o servidor_threads -lpthread && echo "Compilado OK"

In [ ]:
gnome-terminal -- bash -c "./servidor_threads; echo 'Presionar ENTER para cerrar...'; read"

---
## 6) ¿Cómo se cierra una conexión C-S? Métodos para evitar pérdida de información

### Cierre simple con `close()`
```c
close(sockfd);
```
Cierra el socket inmediatamente. Si hay datos en el buffer de envío que aún no fueron transmitidos, pueden perderse.

### Cierre ordenado con `shutdown()`
```c
shutdown(sockfd, SHUT_WR);  // avisa al otro extremo que no enviaremos más datos
// ... esperar a que el otro extremo cierre su lado (recv devuelve 0)
close(sockfd);
```

`shutdown()` permite un **cierre en dos pasos (half-close)**:
- `SHUT_WR`: deja de enviar datos (envía FIN al otro extremo).
- `SHUT_RD`: deja de recibir datos.
- `SHUT_RDWR`: equivalente a `close()` pero sin liberar el descriptor.

### ¿Por qué `shutdown()` evita pérdida de información?

Cuando el cliente llama `shutdown(SHUT_WR)`, el servidor recibe un `recv()` que devuelve `0` (señal de que no hay más datos), pero **aún puede terminar de enviar** su respuesta antes de cerrar. Esto garantiza que ambos lados terminan de transmitir antes de liberar la conexión.

### Secuencia correcta de cierre:
```
Cliente                         Servidor
  shutdown(SHUT_WR)  ──FIN──►
                     ◄──datos pendientes──
  recv() == 0        ◄──FIN──
  close()                         close()
```


---
## 7) Probar el chat cambiando el tipo de socket: TCP → UDP

El chat original (`servidor_chat` / `cliente_chat`) usa `SOCK_STREAM` (TCP). Si cambiamos a `SOCK_DGRAM` (UDP) el comportamiento cambia notablemente:

### Diferencias al cambiar a UDP:

| Aspecto | TCP (SOCK_STREAM) | UDP (SOCK_DGRAM) |
|---|---|---|
| Conexión | `connect()` / `accept()` establece un canal | No hay conexión. Cada datagrama viaja independiente |
| Orden de mensajes | Garantizado | No garantizado |
| Pérdida de mensajes | No hay pérdida (retransmite) | Posible pérdida sin notificación |
| Identificación del cliente | Implícita por el socket del cliente | Debe usarse `recvfrom()` para obtener la dirección |
| API del servidor | `accept()` + `recv()`/`send()` | Solo `recvfrom()` / `sendto()` |

### ¿Qué pasa si probamos el chat con UDP?

- El servidor **no puede saber de antemano** con cuántos clientes va a hablar: no tiene `accept()`.
- No existe el concepto de "cliente conectado": el servidor recibe un datagrama con la IP/puerto del remitente y debe guardar esa info para responderle.
- No hay señal de desconexión: si un cliente cierra, el servidor no lo sabe hasta que deje de recibir datagramas de ese cliente.
- Para un chat multicliente con UDP, el servidor necesita mantener una tabla de clientes conocidos (dirección IP + puerto) y reenviar los mensajes a todos ellos.

El servidor UDP del ejercicio A-2 ya muestra este comportamiento básico.
